In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import find_peaks

# ============================================================
# PATHS
# ============================================================

base_folder = r"C:\Users\cmkua\Downloads\ALANA"
faulty_list_path = r"C:\Users\cmkua\Downloads\Acoustic Wave Data - WIP.csv"

output_folder = os.path.join(base_folder, "onset_recovery_plots")
os.makedirs(output_folder, exist_ok=True)

faulty_file_col = "File Name"

# ============================================================
# LOAD FAULTY FILE LIST
# ============================================================

faulty_df = pd.read_csv(faulty_list_path)
faulty_files = faulty_df[faulty_file_col].dropna().astype(str).tolist()

# ============================================================
# DETECTION SETTINGS TO TEST
# ============================================================

parameter_sets = [
    {
        "setting_name": "original",
        "search_half_width": 0.005,
        "threshold_sigma": 4,
        "height_sigma": 4,
        "prominence_sigma": 3,
        "backtrack_window_s": 0.003,
        "backtrack_sigma": 4,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 0.4
    },
    {
        "setting_name": "wider_3p5sigma",
        "search_half_width": 0.010,
        "threshold_sigma": 3.5,
        "height_sigma": 3.5,
        "prominence_sigma": 2.5,
        "backtrack_window_s": 0.005,
        "backtrack_sigma": 3.5,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 0.8
    },
    {
        "setting_name": "wider_3sigma",
        "search_half_width": 0.010,
        "threshold_sigma": 3,
        "height_sigma": 3,
        "prominence_sigma": 2,
        "backtrack_window_s": 0.006,
        "backtrack_sigma": 3,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 1.0
    },
    {
        "setting_name": "very_wide_soft",
        "search_half_width": 0.015,
        "threshold_sigma": 3,
        "height_sigma": 3,
        "prominence_sigma": 1.5,
        "backtrack_window_s": 0.008,
        "backtrack_sigma": 3,
        "smooth_window_s": 0.00008,
        "flag_lag_ms": 1.2
    }
]

# ============================================================
# FIXED SETTINGS
# ============================================================

expected_shot_time = 0.073
next_shot_start = 0.133

noise_window = 0.003
min_peak_distance_s = 0.010
min_run_s = 0.00005
response_window = 0.010

# ============================================================
# DETECTOR FUNCTION
# ============================================================

def detect_onset_with_settings(x, fs, settings):
    x = x.astype(np.float64)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    abs_x = np.abs(x)

    smooth_n = max(1, int(round(settings["smooth_window_s"] * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")

    search_start = expected_shot_time - settings["search_half_width"]
    search_end = expected_shot_time + settings["search_half_width"]

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    inoise1 = isearch0
    inoise0 = max(0, int(round((search_start - noise_window) * fs)))

    if isearch1 <= isearch0 or inoise1 <= inoise0:
        return None

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    threshold = noise_mean + settings["threshold_sigma"] * noise_std
    min_height = noise_mean + settings["height_sigma"] * noise_std
    min_prominence = settings["prominence_sigma"] * noise_std
    min_peak_distance = int(round(min_peak_distance_s * fs))

    search_env = env[isearch0:isearch1]

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance
    )

    if len(peaks) == 0:
        return {
            "found": False,
            "x": x,
            "env": env,
            "isearch0": isearch0,
            "isearch1": isearch1,
            "threshold": threshold,
            "onset_threshold": np.nan,
            "peak_index": np.nan,
            "onset_index": np.nan,
            "peak_time": np.nan,
            "onset_time": np.nan,
            "lag_ms": np.nan,
            "flagged": True
        }

    best_peak_local = peaks[0]
    peak_index = isearch0 + best_peak_local
    peak_time = peak_index / fs

    backtrack_samples = int(round(settings["backtrack_window_s"] * fs))
    back_start = max(isearch0, peak_index - backtrack_samples)

    noise_back_start = max(0, back_start - int(round(0.005 * fs)))
    noise_back_end = back_start
    local_noise = env[noise_back_start:noise_back_end]

    if len(local_noise) > 0:
        local_noise_mean = np.mean(local_noise)
        local_noise_std = np.std(local_noise)
        onset_threshold = local_noise_mean + settings["backtrack_sigma"] * local_noise_std
    else:
        onset_threshold = threshold

    search_back_env = env[back_start:peak_index]
    above = search_back_env > onset_threshold

    min_run_samples = max(1, int(round(min_run_s * fs)))
    onset_index = None

    for i in range(len(above) - min_run_samples + 1):
        if np.all(above[i:i + min_run_samples]):
            onset_index = back_start + i
            break

    if onset_index is None:
        onset_index = peak_index
        forced_peak = True
    else:
        forced_peak = False

    onset_time = onset_index / fs
    lag_ms = (peak_index - onset_index) / fs * 1000

    response_stop = onset_time + response_window
    truncated = False

    if response_stop > next_shot_start:
        response_stop = next_shot_start
        truncated = True

    flagged = (
        forced_peak
        or truncated
        or lag_ms > settings["flag_lag_ms"]
    )

    return {
        "found": True,
        "x": x,
        "env": env,
        "isearch0": isearch0,
        "isearch1": isearch1,
        "threshold": threshold,
        "onset_threshold": onset_threshold,
        "peak_index": peak_index,
        "onset_index": onset_index,
        "peak_time": peak_time,
        "onset_time": onset_time,
        "lag_ms": lag_ms,
        "flagged": flagged,
        "forced_peak": forced_peak,
        "truncated": truncated
    }

# ============================================================
# RUN EXPERIMENTS
# ============================================================

summary_rows = []

for filename in faulty_files:
    wav_path = os.path.join(base_folder, filename)

    if not os.path.exists(wav_path):
        summary_rows.append({
            "File Name": filename,
            "Setting": "all",
            "Found": False,
            "Notes": "file not found"
        })
        continue

    fs, x_raw = wavfile.read(wav_path)
    full_t = np.arange(len(x_raw)) / fs

    fig, axes = plt.subplots(
        len(parameter_sets),
        1,
        figsize=(12, 3.2 * len(parameter_sets)),
        sharex=True
    )

    if len(parameter_sets) == 1:
        axes = [axes]

    for ax, settings in zip(axes, parameter_sets):
        result = detect_onset_with_settings(x_raw, fs, settings)

        if result is None:
            ax.text(0.5, 0.5, "Invalid window", transform=ax.transAxes,
                    ha="center", va="center")
            continue

        x = result["x"]
        env = result["env"]
        isearch0 = result["isearch0"]
        isearch1 = result["isearch1"]

        ax.plot(full_t[isearch0:isearch1], x[isearch0:isearch1], alpha=0.6, label="waveform")
        ax.plot(full_t[isearch0:isearch1], env[isearch0:isearch1], alpha=0.9, label="envelope")
        ax.axhline(result["threshold"], linestyle=":", label="peak threshold")

        if result["found"]:
            ax.axhline(result["onset_threshold"], linestyle="--", label="onset threshold")
            ax.axvline(result["peak_time"], linestyle=":", label="peak")
            ax.axvline(result["onset_time"], linestyle="--", label="onset")

            title = (
                f"{settings['setting_name']} | "
                f"found=True | flagged={result['flagged']} | "
                f"onset={result['onset_time']:.6f}s | "
                f"lag={result['lag_ms']:.3f} ms"
            )
        else:
            title = f"{settings['setting_name']} | found=False"

        ax.set_title(title)
        ax.set_ylabel("Amplitude")
        ax.legend(loc="upper right", fontsize=8)

        summary_rows.append({
            "File Name": filename,
            "Setting": settings["setting_name"],
            "Found": result["found"],
            "Flagged": result.get("flagged", True),
            "Onset (s)": result.get("onset_time", np.nan),
            "Peak Time (s)": result.get("peak_time", np.nan),
            "Peak-Onset Lag (ms)": result.get("lag_ms", np.nan),
            "Forced Peak": result.get("forced_peak", np.nan),
            "Truncated": result.get("truncated", np.nan)
        })

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(filename, fontsize=12)
    plt.tight_layout()

    safe_name = filename.replace(".wav", "").replace("/", "_").replace("\\", "_")
    plot_path = os.path.join(output_folder, f"{safe_name}_onset_recovery.png")
    plt.savefig(plot_path, dpi=200)
    plt.close(fig)

# ============================================================
# SAVE SUMMARY
# ============================================================

summary_df = pd.DataFrame(summary_rows)

summary_csv = os.path.join(base_folder, "onset_recovery_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print("Saved summary to:")
print(summary_csv)

print("Saved plots to:")
print(output_folder)

print()
print(summary_df.groupby(["Setting", "Found", "Flagged"]).size())

Saved summary to:
C:\Users\cmkua\Downloads\ALANA\onset_recovery_summary.csv
Saved plots to:
C:\Users\cmkua\Downloads\ALANA\onset_recovery_plots

Setting         Found  Flagged
original        False  True       34
                True   True       48
very_wide_soft  False  True       12
                True   False      39
                       True       31
wider_3p5sigma  False  True       29
                True   False      16
                       True       37
wider_3sigma    False  True       26
                True   False      21
                       True       35
dtype: int64


In [ ]:
import pandas as pd

df = pd.read_csv("onset_recovery_summary.csv")

print(
    df.groupby("Setting")
      .agg({
          "Found":"sum",
          "Flagged":lambda x: (~x).sum()
      })
)

In [5]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\cmkua\Downloads\ALANA\onset_recovery_summary.csv"
)



recovered = df[
    (df["Setting"] == "very_wide_soft") &
    (df["Found"] == True) &
    (df["Flagged"] == False)
]

print(len(recovered))

print(recovered["File Name"].tolist()[:20])

39
['scam_0037_0670224376_725_ca0_scam01037_hedgehog_____________02p01.wav', 'scam_0071_0673238205_695_ca0_scam05071_hadahastsaa__________02p01.wav', 'scam_0086_0674573256_865_ca0_scam05086_naatsiilid___________02p01.wav', 'scam_0104_0676170543_585_ca0_scam05104_ad_ees_eez___________01p01.wav', 'scam_0107_0676443644_665_ca0_scam02107_ad_ilidi_____________02p01.wav', 'scam_0110_0676703142_539_ca0_scam03110_naakih_tsaadah_______02p01.wav', 'scam_0112_0676881366_146_ca0_scam02112_tseebii______________01p01.wav', 'scam_0133_0678742642_430_ca0_scam04133_pierrefeu____________01p01.wav', 'scam_0159_0681062056_835_ca0_scam04159_grand_coyer__________02p01.wav', 'scam_0193_0684066469_854_ca0_scam01193_pont_________________01p01.wav', 'scam_0200_0684689393_585_ca0_scam01200_ondres_______________01p01.wav', 'scam_0239_0688152051_670_ca0_scam01239_cheiron______________02p01.wav', 'scam_0239_0688154484_917_ca0_scam02239_content______________01p01.wav', 'scam_0242_0688417845_578_ca0_scam04242_pal____

In [9]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\cmkua\Downloads\Acoustic Wave Data - Recover.csv"
)
print(df["Best Onset"].notna().sum())

43


In [11]:
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import find_peaks, windows
from scipy.stats import linregress

# ============================================================
# PATHS
# ============================================================

base_folder = r"C:\Users\cmkua\Downloads\ALANA"

meta_path = r"C:\Users\cmkua\Downloads\LIBS_acoustic_meta_sheet - LIBS_acoustic_meta_sheet (1).csv"
recover_path = r"C:\Users\cmkua\Downloads\Acoustic Wave Data - Recover.csv"

recovered_output_path = r"C:\Users\cmkua\Downloads\recovered_metrics.csv"
expanded_output_path = r"C:\Users\cmkua\Downloads\LIBS_acoustic_meta_sheet_expanded.csv"

# ============================================================
# SETTINGS
# ============================================================

expected_shot_time = 0.073
next_shot_start = 0.133
response_window = 0.010

fit_db_top = -7
fit_db_bottom = -15
t_c = 0.002

noise_window = 0.003
min_peak_distance_s = 0.010
min_run_s = 0.00005

usable_band = (1000, 50000)
low_band = (1000, 10000)
high_band = (10000, 30000)

parameter_sets = {
    "Original": {
        "search_half_width": 0.005,
        "threshold_sigma": 4,
        "height_sigma": 4,
        "prominence_sigma": 3,
        "backtrack_window_s": 0.003,
        "backtrack_sigma": 4,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 0.4
    },
    "wider 3p5 sigma": {
        "search_half_width": 0.010,
        "threshold_sigma": 3.5,
        "height_sigma": 3.5,
        "prominence_sigma": 2.5,
        "backtrack_window_s": 0.005,
        "backtrack_sigma": 3.5,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 0.8
    },
    "wider 3 sigma": {
        "search_half_width": 0.010,
        "threshold_sigma": 3,
        "height_sigma": 3,
        "prominence_sigma": 2,
        "backtrack_window_s": 0.006,
        "backtrack_sigma": 3,
        "smooth_window_s": 0.00005,
        "flag_lag_ms": 1.0
    },
    "very wide soft": {
        "search_half_width": 0.015,
        "threshold_sigma": 3,
        "height_sigma": 3,
        "prominence_sigma": 1.5,
        "backtrack_window_s": 0.008,
        "backtrack_sigma": 3,
        "smooth_window_s": 0.00008,
        "flag_lag_ms": 1.2
    }
}

# ============================================================
# FUNCTIONS
# ============================================================

def detect_onset(x, fs, settings):
    x = x.astype(np.float64)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    abs_x = np.abs(x)

    smooth_n = max(1, int(round(settings["smooth_window_s"] * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")

    search_start = expected_shot_time - settings["search_half_width"]
    search_end = expected_shot_time + settings["search_half_width"]

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    inoise1 = isearch0
    inoise0 = max(0, int(round((search_start - noise_window) * fs)))

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    min_height = noise_mean + settings["height_sigma"] * noise_std
    min_prominence = settings["prominence_sigma"] * noise_std
    min_peak_distance = int(round(min_peak_distance_s * fs))

    search_env = env[isearch0:isearch1]

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance
    )

    if len(peaks) == 0:
        return None, x

    peak_index = isearch0 + peaks[0]
    peak_time = peak_index / fs

    backtrack_samples = int(round(settings["backtrack_window_s"] * fs))
    back_start = max(isearch0, peak_index - backtrack_samples)

    noise_back_start = max(0, back_start - int(round(0.005 * fs)))
    noise_back_end = back_start
    local_noise = env[noise_back_start:noise_back_end]

    if len(local_noise) > 0:
        onset_threshold = np.mean(local_noise) + settings["backtrack_sigma"] * np.std(local_noise)
    else:
        onset_threshold = noise_mean + settings["threshold_sigma"] * noise_std

    search_back_env = env[back_start:peak_index]
    above = search_back_env > onset_threshold

    min_run_samples = max(1, int(round(min_run_s * fs)))
    onset_index = None

    for i in range(len(above) - min_run_samples + 1):
        if np.all(above[i:i + min_run_samples]):
            onset_index = back_start + i
            break

    if onset_index is None:
        onset_index = peak_index

    onset_time = onset_index / fs
    response_stop = onset_time + response_window

    if response_stop > next_shot_start:
        response_stop = next_shot_start

    lag_ms = (peak_index - onset_index) / fs * 1000

    return {
        "onset_index": onset_index,
        "onset_time": onset_time,
        "peak_time": peak_time,
        "lag_ms": lag_ms,
        "response_stop": response_stop
    }, x


def compute_time_metrics(segment, fs):
    t = np.arange(len(segment)) / fs
    energy = segment ** 2

    edc = np.cumsum(energy[::-1])[::-1]
    edc_norm = edc / np.max(edc)
    edc_db = 10 * np.log10(edc_norm + 1e-20)

    fit_mask = (edc_db <= fit_db_top) & (edc_db >= fit_db_bottom)

    if np.sum(fit_mask) < 2:
        slope = np.nan
        r2 = np.nan
        drop_time = np.nan
    else:
        slope, intercept, r_value, p_value, std_err = linregress(
            t[fit_mask],
            edc_db[fit_mask]
        )
        r2 = r_value ** 2
        db_drop = abs(fit_db_bottom - fit_db_top)
        drop_time = db_drop / abs(slope) if slope != 0 else np.nan

    i_c = int(round(t_c * fs))

    early_energy = np.sum(energy[:i_c])
    late_energy = np.sum(energy[i_c:])

    C2 = 10 * np.log10(early_energy / late_energy) if late_energy > 0 else np.nan

    return slope, r2, drop_time, C2


def compute_fft_metrics(segment, fs):
    segment = segment - np.mean(segment)
    hann = windows.hann(len(segment))
    seg_w = segment * hann

    freqs = np.fft.rfftfreq(len(seg_w), d=1/fs)
    X = np.fft.rfft(seg_w)
    power = np.abs(X) ** 2

    usable_mask = (freqs >= usable_band[0]) & (freqs <= usable_band[1])
    f = freqs[usable_mask]
    p = power[usable_mask]

    if len(p) == 0 or np.sum(p) == 0:
        return {
            "Spectral Centroid (Hz)": np.nan,
            "Spectral Bandwidth (Hz)": np.nan,
            "Peak Frequency (Hz)": np.nan,
            "Rolloff 85% (Hz)": np.nan,
            "Low Power 1-10 kHz": np.nan,
            "High Power 10-30 kHz": np.nan,
            "High/Low Ratio": np.nan,
            "High Frequency Fraction": np.nan,
            "Total FFT Power": np.nan
        }

    p_sum = np.sum(p)

    centroid = np.sum(f * p) / p_sum
    bandwidth = np.sqrt(np.sum(((f - centroid) ** 2) * p) / p_sum)
    peak_freq = f[np.argmax(p)]

    cumulative = np.cumsum(p)
    rolloff_85 = f[np.where(cumulative >= 0.85 * p_sum)[0][0]]

    low_mask = (freqs >= low_band[0]) & (freqs < low_band[1])
    high_mask = (freqs >= high_band[0]) & (freqs < high_band[1])

    low_power = np.sum(power[low_mask])
    high_power = np.sum(power[high_mask])
    total_power = np.sum(power)

    return {
        "Spectral Centroid (Hz)": centroid,
        "Spectral Bandwidth (Hz)": bandwidth,
        "Peak Frequency (Hz)": peak_freq,
        "Rolloff 85% (Hz)": rolloff_85,
        "Low Power 1-10 kHz": low_power,
        "High Power 10-30 kHz": high_power,
        "High/Low Ratio": high_power / low_power if low_power > 0 else np.nan,
        "High Frequency Fraction": high_power / total_power if total_power > 0 else np.nan,
        "Total FFT Power": total_power
    }

# ============================================================
# LOAD SHEETS
# ============================================================

meta_df = pd.read_csv(meta_path)
recover_df = pd.read_csv(recover_path)

recover_df = recover_df.dropna(subset=["Best Onset"]).copy()
recover_df = recover_df[recover_df["Best Onset"].astype(str).str.upper() != "N/A"]

print("Files to recover:", len(recover_df))
print(recover_df["Best Onset"].value_counts())

# ============================================================
# PROCESS RECOVERED FILES
# ============================================================

rows = []

for _, row in recover_df.iterrows():
    filename = row["File Name"]
    method = row["Best Onset"]

    wav_path = os.path.join(base_folder, filename)

    if not os.path.exists(wav_path):
        print("Missing:", filename)
        continue

    settings = parameter_sets[method]

    fs, x_raw = wavfile.read(wav_path)
    detection, x = detect_onset(x_raw, fs, settings)

    if detection is None:
        print("Failed:", filename)
        continue

    i0 = detection["onset_index"]
    i1 = int(round(detection["response_stop"] * fs))
    segment = x[i0:i1]

    if len(segment) == 0:
        print("Empty segment:", filename)
        continue

    slope, r2, drop_time, C2 = compute_time_metrics(segment, fs)
    fft_metrics = compute_fft_metrics(segment, fs)

    rows.append({
        "File Name": filename,
        "Onset (s)": detection["onset_time"],
        "Slope (dB/s)": slope,
        "R^2": r2,
        "Drop Time (s)": drop_time,
        "C2 (dB)": C2,
        "Recovery Method": method,
        **fft_metrics
    })

recovered_metrics = pd.DataFrame(rows)
recovered_metrics.to_csv(recovered_output_path, index=False)

print("Recovered metrics saved to:")
print(recovered_output_path)
print("Recovered rows:", len(recovered_metrics))

# ============================================================
# ADD FORMATION AND MEMBER FROM EXISTING META SHEET IF POSSIBLE
# ============================================================

label_cols = ["File Name", "Formation", "Member"]

labels = meta_df[label_cols].drop_duplicates()

recovered_metrics = recovered_metrics.merge(
    labels,
    on="File Name",
    how="left"
)

# Put Formation and Member near the front
front_cols = [
    "File Name",
    "Onset (s)",
    "Slope (dB/s)",
    "R^2",
    "Drop Time (s)",
    "C2 (dB)",
    "Formation",
    "Member",
    "Recovery Method"
]

other_cols = [c for c in recovered_metrics.columns if c not in front_cols]
recovered_metrics = recovered_metrics[front_cols + other_cols]

# ============================================================
# APPEND TO OLD META SHEET
# ============================================================

expanded_df = pd.concat(
    [meta_df, recovered_metrics],
    ignore_index=True,
    sort=False
)

expanded_df = expanded_df.drop_duplicates(
    subset=["File Name"],
    keep="first"
)

expanded_df.to_csv(expanded_output_path, index=False)

print("Expanded meta sheet saved to:")
print(expanded_output_path)
print("Old rows:", len(meta_df))
print("Recovered rows added:", len(recovered_metrics))
print("Expanded rows:", len(expanded_df))

Files to recover: 43
Best Onset
Original           21
wider 3 sigma      12
very wide soft      7
wider 3p5 sigma     3
Name: count, dtype: int64
Recovered metrics saved to:
C:\Users\cmkua\Downloads\recovered_metrics.csv
Recovered rows: 43
Expanded meta sheet saved to:
C:\Users\cmkua\Downloads\LIBS_acoustic_meta_sheet_expanded.csv
Old rows: 127
Recovered rows added: 43
Expanded rows: 170


In [13]:
import pandas as pd

df = pd.read_csv(r"C:\Users\cmkua\Downloads\LIBS_acoustic_meta_sheet_expanded.csv")

print(df["Member"].value_counts())

Member
Undivided    41
Nataani      38
Rochette     24
Bastide      22
Roubion      11
Caille       10
Cha'al        9
Content       8
Artuby        7
Name: count, dtype: int64


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import find_peaks

# ============================================================
# PATHS
# ============================================================

base_folder = r"C:\Users\cmkua\Downloads\ALANA"

output_folder = r"C:\Users\cmkua\Downloads\recovery_pass_2_plots_v2"
os.makedirs(output_folder, exist_ok=True)

summary_path = r"C:\Users\cmkua\Downloads\recovery_pass_2_summary_v2.csv"

# ============================================================
# REMAINING 39 FILES
# ============================================================

faulty_files = [
    "scam_0071_0673238158_718_ca0_scam05071_hadahastsaa__________01p01.wav",
    "scam_0089_0674854082_617_ca0_scam01089_taa-ihaaih___________01p01.wav",
    "scam_0104_0676170543_585_ca0_scam05104_ad_ees_eez___________01p01.wav",
    "scam_0110_0676703142_539_ca0_scam03110_naakih_tsaadah_______02p01.wav",
    "scam_0113_0676966786_559_ca0_scam01113_mussih_______________01p01.wav",
    "scam_0113_0676966938_600_ca0_scam01113_mussih_______________02p01.wav",
    "scam_0131_0678564646_684_ca0_scam04131_cheval_blanc_________02p01.wav",
    "scam_0147_0679984828_603_ca0_scam04147_pepin________________01p01.wav",
    "scam_0170_0682026345_821_ca0_scam01170_lattes_______________02p01.wav",
    "scam_0183_0683183544_821_ca0_scam01183_sauzeries_hautes_____01p01.wav",
    "scam_0183_0683183736_826_ca0_scam01183_sauzeries_hautes_____02p01.wav",
    "scam_0183_0683186320_932_ca0_scam02183_sauzeries_basses_____01p01.wav",
    "scam_0183_0683186485_901_ca0_scam02183_sauzeries_basses_____02p01.wav",
    "scam_0184_0683272998_853_ca0_scam03184_sauze________________02p01.wav",
    "scam_0211_0685666540_589_ca0_scam04211_penne________________02p01.wav",
    "scam_0213_0685844643_151_ca0_scam04213_moustiers_sainte_mar_01p01.wav",
    "scam_0239_0688151857_681_ca0_scam01239_cheiron______________01p01.wav",
    "scam_0242_0688417845_578_ca0_scam04242_pal__________________01p01.wav",
    "scam_0242_0688418039_588_ca0_scam04242_pal__________________02p01.wav",
    "scam_0242_0688420559_670_ca0_scam05242_nans_________________01p01.wav",
    "scam_0242_0688420722_703_ca0_scam05242_nans_________________02p01.wav",
    "scam_0246_0688773232_591_ca0_scam01246_amignon______________01p01.wav",
    "scam_0250_0689128907_101_ca0_scam01250_hotel________________02p01.wav",
    "scam_0250_0689145560_077_ca0_scam04250_villeneuve___________01p01.wav",
    "scam_0250_0689145702_059_ca0_scam04250_villeneuve___________02p01.wav",
    "scam_0274_0691261308_479_ca0_scam01274_melle________________01p01.wav",
    "scam_0274_0691261499_530_ca0_scam01274_melle________________02p01.wav",
    "scam_0274_0691263225_224_ca0_scam03274_chasteuil____________01p01.wav",
    "scam_0286_0692324948_436_ca0_scam01286_bezaudun_____________01p01.wav",
    "scam_0309_0694365855_967_ca0_scam04309_riez_________________01p01.wav",
    "scam_0312_0694632787_748_ca0_scam01312_riolan_312___________02p01.wav",
    "scam_0313_0694724321_603_ca0_scam01313_bessons______________01p01.wav",
    "scam_0328_0696058027_273_ca0_scam05328_brusquet_____________02p01.wav",
    "scam_0335_0696680688_405_ca0_scam01335_chabran______________01p01.wav",
    "scam_0335_0696680738_320_ca0_scam01335_chabran______________02p01.wav",
    "scam_0343_0697389227_661_ca0_scam02343_chanolles____________01p01.wav",
    "scam_0343_0697389276_654_ca0_scam02343_chanolles____________02p01.wav",
    "scam_0361_0698983318_603_ca0_scam01361_naanazwod____________01p01.wav",
    "scam_0361_0698983364_602_ca0_scam01361_naanazwod____________02p01.wav",
]

# ============================================================
# PASS 2 SETTINGS
# ============================================================

expected_times = [0.060, 0.065, 0.070, 0.073, 0.076, 0.080, 0.085]

settings = {
    "search_half_width": 0.008,
    "threshold_sigma": 3.5,
    "height_sigma": 3.5,
    "prominence_sigma": 2.5,
    "backtrack_window_s": 0.006,
    "backtrack_sigma": 3.5,
    "smooth_window_s": 0.00005,
    "flag_lag_ms": 1.0
}

noise_window = 0.003
min_peak_distance_s = 0.010
min_run_s = 0.00005

# ============================================================
# DETECTOR
# ============================================================

def prepare_audio(x_raw):
    x = x_raw.astype(np.float64)
    if x.ndim > 1:
        x = x.mean(axis=1)
    x = x - np.mean(x)
    return x


def make_envelope(x, fs, smooth_window_s):
    abs_x = np.abs(x)
    smooth_n = max(1, int(round(smooth_window_s * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")
    return env


def detect_onset(x, env, fs, settings, expected_shot_time):
    search_start = expected_shot_time - settings["search_half_width"]
    search_end = expected_shot_time + settings["search_half_width"]

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    inoise1 = isearch0
    inoise0 = max(0, int(round((search_start - noise_window) * fs)))

    if isearch1 <= isearch0 or inoise1 <= inoise0:
        return None

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    peak_threshold = noise_mean + settings["threshold_sigma"] * noise_std
    min_height = noise_mean + settings["height_sigma"] * noise_std
    min_prominence = settings["prominence_sigma"] * noise_std
    min_peak_distance = int(round(min_peak_distance_s * fs))

    search_env = env[isearch0:isearch1]

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance
    )

    result = {
        "found": False,
        "isearch0": isearch0,
        "isearch1": isearch1,
        "peak_threshold": peak_threshold,
        "onset_threshold": np.nan,
        "peak_time": np.nan,
        "onset_time": np.nan,
        "lag_ms": np.nan,
        "flagged": True,
        "forced_peak": np.nan
    }

    if len(peaks) == 0:
        return result

    peak_index = isearch0 + peaks[0]
    peak_time = peak_index / fs

    backtrack_samples = int(round(settings["backtrack_window_s"] * fs))
    back_start = max(isearch0, peak_index - backtrack_samples)

    noise_back_start = max(0, back_start - int(round(0.005 * fs)))
    noise_back_end = back_start
    local_noise = env[noise_back_start:noise_back_end]

    if len(local_noise) > 0:
        onset_threshold = np.mean(local_noise) + settings["backtrack_sigma"] * np.std(local_noise)
    else:
        onset_threshold = peak_threshold

    search_back_env = env[back_start:peak_index]
    above = search_back_env > onset_threshold

    min_run_samples = max(1, int(round(min_run_s * fs)))
    onset_index = None

    for i in range(len(above) - min_run_samples + 1):
        if np.all(above[i:i + min_run_samples]):
            onset_index = back_start + i
            break

    forced_peak = False
    if onset_index is None:
        onset_index = peak_index
        forced_peak = True

    onset_time = onset_index / fs
    lag_ms = (peak_index - onset_index) / fs * 1000
    flagged = forced_peak or lag_ms > settings["flag_lag_ms"]

    result.update({
        "found": True,
        "onset_index": onset_index,
        "peak_index": peak_index,
        "peak_time": peak_time,
        "onset_time": onset_time,
        "lag_ms": lag_ms,
        "flagged": flagged,
        "forced_peak": forced_peak,
        "onset_threshold": onset_threshold
    })

    return result

# ============================================================
# PLOTTING
# ============================================================

def plot_detection_panel(ax, full_t, x, env, fs, result, expected_time):
    display_start = max(0, expected_time - 0.020)
    display_end = min(len(x) / fs, expected_time + 0.020)

    id0 = int(round(display_start * fs))
    id1 = int(round(display_end * fs))

    ax.plot(full_t[id0:id1], x[id0:id1], alpha=0.5, linewidth=0.8, label="waveform")
    ax.plot(full_t[id0:id1], env[id0:id1], alpha=0.95, linewidth=1.2, label="envelope")

    ax.axvline(expected_time, linestyle="-.", linewidth=1.0, label="expected")

    if result is not None:
        ax.axvspan(result["isearch0"] / fs, result["isearch1"] / fs, alpha=0.12)
        ax.axhline(result["peak_threshold"], linestyle=":", linewidth=1.0)

        if result["found"]:
            ax.axhline(result["onset_threshold"], linestyle="--", linewidth=1.0)
            ax.axvline(result["peak_time"], linestyle=":", linewidth=1.5)

            if result["flagged"]:
                onset_color = "red"
            else:
                onset_color = "green"

            ax.axvline(result["onset_time"], color=onset_color, linewidth=2.5)

            title = (
                f"Expected {expected_time:.3f}s\n"
                f"onset {result['onset_time']:.6f}s, "
                f"lag {result['lag_ms']:.2f} ms, "
                f"flag {result['flagged']}"
            )
        else:
            title = f"Expected {expected_time:.3f}s\nno peak found"
    else:
        title = f"Expected {expected_time:.3f}s\ninvalid window"

    ax.set_title(title, fontsize=9)
    ax.set_xlim(display_start, display_end)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")


summary_rows = []

for filename in faulty_files:
    wav_path = os.path.join(base_folder, filename)

    if not os.path.exists(wav_path):
        print("Missing:", filename)
        continue

    fs, x_raw = wavfile.read(wav_path)

    x = prepare_audio(x_raw)
    env = make_envelope(x, fs, settings["smooth_window_s"])
    full_t = np.arange(len(x)) / fs

    fig, axes = plt.subplots(4, 2, figsize=(15, 13), sharey=True)
    axes = axes.flatten()

    for i, expected_time in enumerate(expected_times):
        ax = axes[i]

        result = detect_onset(
            x=x,
            env=env,
            fs=fs,
            settings=settings,
            expected_shot_time=expected_time
        )

        plot_detection_panel(
            ax=ax,
            full_t=full_t,
            x=x,
            env=env,
            fs=fs,
            result=result,
            expected_time=expected_time
        )

        if result is None:
            summary_rows.append({
                "File Name": filename,
                "Expected Time (s)": expected_time,
                "Found": False,
                "Flagged": True,
                "Onset (s)": np.nan,
                "Peak Time (s)": np.nan,
                "Lag (ms)": np.nan,
                "Forced Peak": np.nan
            })
        else:
            summary_rows.append({
                "File Name": filename,
                "Expected Time (s)": expected_time,
                "Found": result["found"],
                "Flagged": result["flagged"],
                "Onset (s)": result["onset_time"],
                "Peak Time (s)": result["peak_time"],
                "Lag (ms)": result["lag_ms"],
                "Forced Peak": result["forced_peak"]
            })

    axes[-1].axis("off")

    fig.suptitle(filename, fontsize=13)
    plt.tight_layout(rect=[0, 0.02, 1, 0.96])

    safe_name = filename.replace(".wav", "")
    save_path = os.path.join(output_folder, f"{safe_name}_pass2_v2.png")

    plt.savefig(save_path, dpi=220)
    plt.close(fig)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_path, index=False)

print("Saved plots to:")
print(output_folder)
print("Saved summary to:")
print(summary_path)
print()
print(summary_df.groupby(["Expected Time (s)", "Found", "Flagged"]).size())

Saved plots to:
C:\Users\cmkua\Downloads\recovery_pass_2_plots_v2
Saved summary to:
C:\Users\cmkua\Downloads\recovery_pass_2_summary_v2.csv

Expected Time (s)  Found  Flagged
0.060              False  True        3
                   True   False      14
                          True       22
0.065              False  True       21
                   True   False       7
                          True       11
0.070              False  True       27
                   True   False       5
                          True        7
0.073              False  True       22
                   True   False       6
                          True       11
0.076              False  True       14
                   True   False       9
                          True       16
0.080              False  True        6
                   True   False      16
                          True       17
0.085              False  True       11
                   True   False      18
                         

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import find_peaks, windows
from scipy.stats import linregress

# ============================================================
# PATHS
# ============================================================

base_folder = r"C:\Users\cmkua\Downloads\ALANA"

meta_path = r"C:\Users\cmkua\Downloads\LIBS_acoustic_meta_sheet_expanded.csv"

pass2_recovered_output_path = r"C:\Users\cmkua\Downloads\recovered_metrics_pass2.csv"
final_output_path = r"C:\Users\cmkua\Downloads\LIBS_acoustic_meta_sheet_v3_185.csv"

# ============================================================
# PASS 2 ACCEPTED FILES
# ============================================================

pass2_recoveries = [
    ("scam_0071_0673238158_718_ca0_scam05071_hadahastsaa__________01p01.wav", 0.070),
    ("scam_0104_0676170543_585_ca0_scam05104_ad_ees_eez___________01p01.wav", 0.085),
    ("scam_0113_0676966786_559_ca0_scam01113_mussih_______________01p01.wav", 0.085),
    ("scam_0113_0676966938_600_ca0_scam01113_mussih_______________02p01.wav", 0.085),
    ("scam_0183_0683183544_821_ca0_scam01183_sauzeries_hautes_____01p01.wav", 0.085),
    ("scam_0183_0683183736_826_ca0_scam01183_sauzeries_hautes_____02p01.wav", 0.085),
    ("scam_0213_0685844643_151_ca0_scam04213_moustiers_sainte_mar_01p01.wav", 0.076),
    ("scam_0250_0689128907_101_ca0_scam01250_hotel________________02p01.wav", 0.085),
    ("scam_0274_0691263225_224_ca0_scam03274_chasteuil____________01p01.wav", 0.085),
    ("scam_0286_0692324948_436_ca0_scam01286_bezaudun_____________01p01.wav", 0.085),
    ("scam_0312_0694632787_748_ca0_scam01312_riolan_312___________02p01.wav", 0.080),
    ("scam_0335_0696680688_405_ca0_scam01335_chabran______________01p01.wav", 0.085),
    ("scam_0335_0696680738_320_ca0_scam01335_chabran______________02p01.wav", 0.085),
    ("scam_0343_0697389227_661_ca0_scam02343_chanolles____________01p01.wav", 0.085),
    ("scam_0361_0698983364_602_ca0_scam01361_naanazwod____________02p01.wav", 0.076),
]

# ============================================================
# SETTINGS
# ============================================================

next_shot_start = 0.133
response_window = 0.010

fit_db_top = -7
fit_db_bottom = -15
t_c = 0.002

noise_window = 0.003
min_peak_distance_s = 0.010
min_run_s = 0.00005

usable_band = (1000, 50000)
low_band = (1000, 10000)
high_band = (10000, 30000)

# Pass 2 detector settings
settings = {
    "search_half_width": 0.008,
    "threshold_sigma": 3.5,
    "height_sigma": 3.5,
    "prominence_sigma": 2.5,
    "backtrack_window_s": 0.006,
    "backtrack_sigma": 3.5,
    "smooth_window_s": 0.00005,
    "flag_lag_ms": 1.0
}

# ============================================================
# FUNCTIONS
# ============================================================

def prepare_audio(x_raw):
    x = x_raw.astype(np.float64)

    if x.ndim > 1:
        x = x.mean(axis=1)

    x = x - np.mean(x)
    return x


def make_envelope(x, fs, smooth_window_s):
    abs_x = np.abs(x)
    smooth_n = max(1, int(round(smooth_window_s * fs)))
    kernel = np.ones(smooth_n) / smooth_n
    env = np.convolve(abs_x, kernel, mode="same")
    return env


def detect_onset(x, env, fs, settings, expected_shot_time):
    search_start = expected_shot_time - settings["search_half_width"]
    search_end = expected_shot_time + settings["search_half_width"]

    isearch0 = max(0, int(round(search_start * fs)))
    isearch1 = min(len(x), int(round(search_end * fs)))

    inoise1 = isearch0
    inoise0 = max(0, int(round((search_start - noise_window) * fs)))

    if isearch1 <= isearch0 or inoise1 <= inoise0:
        return None

    noise_env = env[inoise0:inoise1]
    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    peak_threshold = noise_mean + settings["threshold_sigma"] * noise_std
    min_height = noise_mean + settings["height_sigma"] * noise_std
    min_prominence = settings["prominence_sigma"] * noise_std
    min_peak_distance = int(round(min_peak_distance_s * fs))

    search_env = env[isearch0:isearch1]

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance
    )

    if len(peaks) == 0:
        return None

    peak_index = isearch0 + peaks[0]
    peak_time = peak_index / fs

    backtrack_samples = int(round(settings["backtrack_window_s"] * fs))
    back_start = max(isearch0, peak_index - backtrack_samples)

    noise_back_start = max(0, back_start - int(round(0.005 * fs)))
    noise_back_end = back_start
    local_noise = env[noise_back_start:noise_back_end]

    if len(local_noise) > 0:
        onset_threshold = np.mean(local_noise) + settings["backtrack_sigma"] * np.std(local_noise)
    else:
        onset_threshold = peak_threshold

    search_back_env = env[back_start:peak_index]
    above = search_back_env > onset_threshold

    min_run_samples = max(1, int(round(min_run_s * fs)))
    onset_index = None

    for i in range(len(above) - min_run_samples + 1):
        if np.all(above[i:i + min_run_samples]):
            onset_index = back_start + i
            break

    forced_peak = False

    if onset_index is None:
        onset_index = peak_index
        forced_peak = True

    onset_time = onset_index / fs
    response_stop = onset_time + response_window

    if response_stop > next_shot_start:
        response_stop = next_shot_start

    lag_ms = (peak_index - onset_index) / fs * 1000

    flagged = forced_peak or lag_ms > settings["flag_lag_ms"]

    return {
        "onset_index": onset_index,
        "onset_time": onset_time,
        "peak_time": peak_time,
        "peak_onset_lag_ms": lag_ms,
        "response_stop": response_stop,
        "flagged": flagged,
        "forced_peak": forced_peak
    }


def compute_time_metrics(segment, fs):
    t = np.arange(len(segment)) / fs
    energy = segment ** 2

    edc = np.cumsum(energy[::-1])[::-1]

    if np.max(edc) == 0:
        return np.nan, np.nan, np.nan, np.nan

    edc_norm = edc / np.max(edc)
    edc_db = 10 * np.log10(edc_norm + 1e-20)

    fit_mask = (edc_db <= fit_db_top) & (edc_db >= fit_db_bottom)

    if np.sum(fit_mask) < 2:
        slope = np.nan
        r2 = np.nan
        drop_time = np.nan
    else:
        slope, intercept, r_value, p_value, std_err = linregress(
            t[fit_mask],
            edc_db[fit_mask]
        )
        r2 = r_value ** 2
        db_drop = abs(fit_db_bottom - fit_db_top)
        drop_time = db_drop / abs(slope) if slope != 0 else np.nan

    i_c = int(round(t_c * fs))

    if i_c >= len(segment):
        return slope, r2, drop_time, np.nan

    early_energy = np.sum(energy[:i_c])
    late_energy = np.sum(energy[i_c:])

    C2 = 10 * np.log10(early_energy / late_energy) if late_energy > 0 else np.nan

    return slope, r2, drop_time, C2


def compute_fft_metrics(segment, fs):
    segment = segment - np.mean(segment)

    hann = windows.hann(len(segment))
    seg_w = segment * hann

    freqs = np.fft.rfftfreq(len(seg_w), d=1/fs)
    X = np.fft.rfft(seg_w)
    power = np.abs(X) ** 2

    usable_mask = (freqs >= usable_band[0]) & (freqs <= usable_band[1])
    f = freqs[usable_mask]
    p = power[usable_mask]

    if len(p) == 0 or np.sum(p) == 0:
        return {
            "Spectral Centroid (Hz)": np.nan,
            "Spectral Bandwidth (Hz)": np.nan,
            "Peak Frequency (Hz)": np.nan,
            "Rolloff 85% (Hz)": np.nan,
            "Low Power 1-10 kHz": np.nan,
            "High Power 10-30 kHz": np.nan,
            "High/Low Ratio": np.nan,
            "High Frequency Fraction": np.nan,
            "Total FFT Power": np.nan
        }

    p_sum = np.sum(p)

    centroid = np.sum(f * p) / p_sum
    bandwidth = np.sqrt(np.sum(((f - centroid) ** 2) * p) / p_sum)
    peak_freq = f[np.argmax(p)]

    cumulative = np.cumsum(p)
    rolloff_85 = f[np.where(cumulative >= 0.85 * p_sum)[0][0]]

    low_mask = (freqs >= low_band[0]) & (freqs < low_band[1])
    high_mask = (freqs >= high_band[0]) & (freqs < high_band[1])

    low_power = np.sum(power[low_mask])
    high_power = np.sum(power[high_mask])
    total_power = np.sum(power)

    return {
        "Spectral Centroid (Hz)": centroid,
        "Spectral Bandwidth (Hz)": bandwidth,
        "Peak Frequency (Hz)": peak_freq,
        "Rolloff 85% (Hz)": rolloff_85,
        "Low Power 1-10 kHz": low_power,
        "High Power 10-30 kHz": high_power,
        "High/Low Ratio": high_power / low_power if low_power > 0 else np.nan,
        "High Frequency Fraction": high_power / total_power if total_power > 0 else np.nan,
        "Total FFT Power": total_power
    }

# ============================================================
# PROCESS PASS 2 FILES
# ============================================================

rows = []

for filename, expected_time in pass2_recoveries:
    wav_path = os.path.join(base_folder, filename)

    if not os.path.exists(wav_path):
        print("Missing:", filename)
        continue

    fs, x_raw = wavfile.read(wav_path)

    x = prepare_audio(x_raw)
    env = make_envelope(x, fs, settings["smooth_window_s"])

    detection = detect_onset(
        x=x,
        env=env,
        fs=fs,
        settings=settings,
        expected_shot_time=expected_time
    )

    if detection is None:
        print("Failed detection:", filename)
        continue

    i0 = detection["onset_index"]
    i1 = int(round(detection["response_stop"] * fs))

    segment = x[i0:i1]

    if len(segment) == 0:
        print("Empty segment:", filename)
        continue

    slope, r2, drop_time, C2 = compute_time_metrics(segment, fs)
    fft_metrics = compute_fft_metrics(segment, fs)

    rows.append({
        "File Name": filename,
        "Flagged": detection["flagged"],
        "Sample Rate (Hz)": fs,
        "Onset (s)": detection["onset_time"],
        "Peak Time (s)": detection["peak_time"],
        "Peak-Onset Lag (ms)": detection["peak_onset_lag_ms"],
        "Slope (dB/s)": slope,
        "R^2": r2,
        "Drop Time (s)": drop_time,
        "C2 (dB)": C2,
        "Recovery Method": "pass2_expected_time_sweep",
        "Pass2 Expected Time (s)": expected_time,
        **fft_metrics
    })

pass2_df = pd.DataFrame(rows)
pass2_df.to_csv(pass2_recovered_output_path, index=False)

print("Pass 2 recovered metrics saved to:")
print(pass2_recovered_output_path)
print("Pass 2 rows:", len(pass2_df))

# ============================================================
# APPEND TO EXISTING META SHEET
# ============================================================

meta_df = pd.read_csv(meta_path)

combined_df = pd.concat(
    [meta_df, pass2_df],
    ignore_index=True,
    sort=False
)

combined_df = combined_df.drop_duplicates(
    subset=["File Name"],
    keep="first"
)

combined_df.to_csv(final_output_path, index=False)

print()
print("Final meta sheet saved to:")
print(final_output_path)
print("Old rows:", len(meta_df))
print("Pass 2 rows added:", len(pass2_df))
print("Final rows:", len(combined_df))